# 05 — Optional FP32 generation and KV-cache sanity check
Independent host-side experiment. Loads the full PyTorch model and generates 20 tokens
after a 32-token prompt. This does not run quantized C inference.
The block-0 hook should see one 32-token prefill and 19 one-token decode calls.
The first generated token comes from the prefill call, so 20 generated tokens require 20 calls here.

In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/python/workflow_config.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter in the extracted tinystories-rv64imc folder.')
sys.path.insert(0, str(ROOT / 'src/python'))
from workflow_config import (MODEL_ID, MODEL_REVISION, SEQUENCE_LENGTH,
                             MODEL_DIR, EXPORT_DIR, RESULT_DIR, C_DIR, MODEL_PATH,
                             PROMPT, GENERATED_LENGTH)
from notebook_runs import start_run, finish_run
RUN_DIR = start_run(ROOT, 'experiments/generation_kv_cache/01_generate.ipynb',
                    {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'prompt_tokens': SEQUENCE_LENGTH, 'generated_tokens': GENERATED_LENGTH})


In [ ]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
def load_tinystories_model(model_id, revision, attn_implementation="eager"):
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision)

    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, attn_implementation=attn_implementation)

    model.eval()

    return model, tokenizer

In [ ]:
model, tokenizer = load_tinystories_model(MODEL_ID, MODEL_REVISION)
PROMPT_LENGTH = SEQUENCE_LENGTH
encoded = tokenizer(PROMPT, add_special_tokens=False, return_tensors='pt')
available_tokens = encoded['input_ids'].shape[1]

In [ ]:
if available_tokens < PROMPT_LENGTH:
    raise ValueError(
        f"Prompt contains only {available_tokens} tokens. "
        f"At least {PROMPT_LENGTH} are required."
    )

input_ids = encoded["input_ids"][:, :PROMPT_LENGTH]
attention_mask = torch.ones_like(input_ids)

print("Benchmark input shape:", input_ids.shape)
print("Benchmark token count:", input_ids.shape[1])
print("Token IDs:", input_ids[0].tolist())
print("Truncated prompt:", tokenizer.decode(input_ids[0]))

assert input_ids.shape == (1, 32)

In [ ]:
block_input_lengths = []

def record_block_input_length(module, args):
    hidden_states = args[0]

    block_input_lengths.append(
        hidden_states.shape[1]
    )

hook_handle = (
    model.transformer.h[0]
    .register_forward_pre_hook(
        record_block_input_length
    )
)

In [ ]:
try:
    with torch.no_grad():
        generated_sequences = model.generate(
            input_ids=input_ids, attention_mask=attention_mask,
            min_new_tokens=GENERATED_LENGTH, max_new_tokens=GENERATED_LENGTH,
            do_sample=False, use_cache=True, pad_token_id=tokenizer.eos_token_id)
finally:
    hook_handle.remove()

In [ ]:
generated_token_ids = generated_sequences[
    :,
    PROMPT_LENGTH:
]

print("Prompt tokens:", input_ids.shape[1])
print("Generated tokens:", generated_token_ids.shape[1])
print("Total tokens:", generated_sequences.shape[1])

assert input_ids.shape[1] == 32
assert generated_token_ids.shape[1] == 20
assert generated_sequences.shape[1] == 52

In [ ]:
print(
    "Generated continuation:",
    tokenizer.decode(
        generated_token_ids[0],
        skip_special_tokens=True,
    ),
)

print(
    "Complete sequence:",
    tokenizer.decode(
        generated_sequences[0],
        skip_special_tokens=True,
    ),
)

In [ ]:
assert block_input_lengths[0] == 32
assert len(block_input_lengths) == 20
assert all(
    length == 1
    for length in block_input_lengths[1:]
)

print("KV-cache behavior verified")

In [ ]:
import json
report = {'input_ids': input_ids.tolist(), 'generated_ids': generated_token_ids.tolist(),
          'block_input_lengths': block_input_lengths,
          'continuation': tokenizer.decode(generated_token_ids[0], skip_special_tokens=True)}
path = RUN_DIR / 'generation.json'
path.write_text(json.dumps(report, indent=2))
finish_run(ROOT, RUN_DIR, [path])